# The paper's APPENDIX tables and figures

The companion of `build_paper_tables_and_figs.ipynb` for the appendix. Same contract: this notebook sets the run
lists and calls one function per table or figure; all selection and drawing live in `pim.figures.tables`, every
number is read from files written by the canonical scripts, nothing is computed here, and every table is drawn as a
figure in the master tables' style (Othello above discworld, a heavy rule between them).

## Table A1 — held-out predictive loss between the trivial predictor and the Bayes floor

*Are the models we probe and edit good predictors of their environments?* Each run's held-out loss, placed between
two model-free reference points of its environment instance: the **trivial predictor** (the best it could do while
ignoring the history) and the **Bayes floor** (the best ANY predictor could do from the same history). Panel (a) holds
the three; panel (b) the excess over the floor, coloured by its size relative to the floor — the one scale comparable
between Othello's nats and Rayworld's squared intensities.

| quantity | definition | where it comes from |
|---|---|---|
| **trivial predictor** | the best history-blind CONSTANT, fitted on the instance's probe corpus (disjoint from the held-out split — the same rule as Probe Skill's trivial predictor) and scored on the same held-out sequences as the model: Rayworld MSE — the per-ray mean frame (its loss is the per-ray variance of the observation); Othello CE — the move frequencies (≈ uniform over the 60 squares, log 60 = 4.094); token CE — the frequencies of whole-frame patterns. | `runs/_baselines/<instance>/bayes_floor.json` → `trivial` (`pim.environments.discworld.bayes.trivial_predictors`, `pim.environments.othello.bayes.trivial_ce`) |
| **loss** | the run's training objective on the instance's held-out split, averaged as in training: Othello — cross-entropy of the next move, nats per move; Rayworld — squared error of the next frame, mean over positions and rays (intensity² per ray). For Rayworld it is taken on the same sequences the floor was estimated on (paired). Its `±` = SD over the run's seed replicates, when they are scored. | `runs/<topic>/<run>/scores.json` → `prediction` (`scripts/score_prediction.py` → `pim.environments.prediction.score_run`) |
| **Bayes floor — Othello (exact, no ±)** | the generator draws uniformly from the legal set and the history determines the board, so the optimal predictor is uniform over legal moves: floor = E[log \|legal\|] over held-out positions. | `runs/_baselines/<instance>/bayes_floor.json` (`scripts/bayes_floor.py` → `pim.environments.othello.bayes`) |
| **Bayes floor — Rayworld (estimated, value ± uncertainty)** | rendering is deterministic, so the only uncertainty is the discs' state. The posterior over the initial state given frames 0..t (generator prior × the generator's trajectory acceptance × exact agreement with the observed frames) is sampled. Two estimators of the floor come out of the same samples: `lo`, the posterior variance of frame t+1 (biased low if the sampler under-explores), and `hi`, the error of the sampler's own mean prediction against the true next frame (an achievable predictor, so never below the floor in expectation); both are means over positions and rays. **The value shown is their midpoint; the ± is half their distance plus one standard error over sequences** — it covers both estimators and their sampling noise. | same file (`pim.environments.discworld.bayes`); `pim.metrics.prediction.floor_estimate` |
| **excess** | loss − floor, with the floor's ±; in brackets, the excess as a percentage of the floor (the colour). | `pim.metrics.prediction.excess_estimate` |
| *gap closed* (printed in cell [3], not drawn) | (trivial − loss) / (trivial − floor): 1 at the floor, 0 at the trivial predictor. | `pim.metrics.prediction.gap_closed` |

The frames-as-tokens run has two rows: its own objective (cross-entropy over the frame vocabulary, nats per frame —
the Othello reading, with the floor of that same cross-entropy from the same posterior samples) and its output
collapsed to the **mean frame** Σ p(v)·frame(v), scored like a frame model against the same MSE floor as
`L-dw-8ray-20m`.

⚠ On Rayworld the constant predictor is a very weak reference: discs move 0.05–0.12 units a frame, so *repeating the
current frame* is already close to the floor (dw-8ray: constant 0.101, repeat-last-frame 0.0078, floor ≈ 0.0056). That
naive predictor is recorded in each floor file as `trivial.persistence_mse` and printed in cell [3]; it is not a
"trivial predictor" in the constant sense, so it is not a column.

A row shows `—` until its file exists: run `scripts/bayes_floor.py` (floors + trivial predictors) and
`scripts/score_prediction.py` (losses), then re-execute this notebook.

In [ ]:
# [1] THE RUN LISTS — the only thing to edit. Othello first, then discworld; the order here is the row order.
RUNS_OTH = ['L-oth-20m', 'L-oth-adjacent-flip-20m', 'L-oth-adjacent-20m', 'L-oth-noflip-20m']
RUNS_DW = ['L-dw-noiseless-20m', 'L-dw-blink-20m', 'L-dw-128ray-20m', 'L-dw-16ray-20m', 'L-dw-8ray-20m',
           'L-dw-8ray-tok-20m', 'L-dw-5ray-20m']          # the token run sits beside its 8-ray sibling (same instance)

import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
from pim.figures import tables as T

# TITLES: none on the panels in THIS notebook — the paper's captions carry them.
T.SHOW_TITLES = False

# Seed replicates (<run>__seed<k>) fold into the ± on the loss, pooled at a matched budget exactly as in the
# main tables (the replicate sets come from the same collect call).
F = T.collect(RUNS_OTH, RUNS_DW, label='appendix')
print(f'{len(F.df)} (run, target) rows · replicate cells {len(F.rep_sd)}' + (f' · MISSING {F.missing}' if F.missing else ''))

In [ ]:
# [2] TABLE A1 — held-out predictive loss · Bayes floor · excess
fig = T.table_prediction(RUNS_OTH, RUNS_DW, 'A1', rep_sd=F.rep_sd)
plt.show() if fig else print('Table A1: nothing to draw')

In [ ]:
# [3] The same numbers, printed (for reading without the figure; floor_lo / floor_hi are the raw bracket behind a
#     sampled floor's ±; gap_closed = (trivial − loss) / (trivial − floor)) + per Rayworld floor: the naive
#     repeat-the-current-frame predictor and the sampler's diagnostics
import json

P = T.prediction_rows(RUNS_OTH, RUNS_DW, F.rep_sd)
print(P[['run', 'reading', 'trivial', 'loss', 'loss_sd', 'floor', 'floor_pm', 'floor_lo', 'floor_hi', 'excess',
         'excess_pm', 'excess_rel', 'gap_closed', 'n_paired']].to_string(index=False))
for inst in dict.fromkeys(P['instance']):
    fp = REPO / 'runs' / '_baselines' / inst / 'bayes_floor.json'
    if fp.exists() and 'diagnostics' in (b := json.loads(fp.read_text())):
        d, c = b['diagnostics'], b.get('check_position_0', {})
        pers = (b.get('trivial') or {}).get('persistence_mse', {}).get('value', float('nan'))
        print(f"{inst}: repeat-last-frame MSE {pers:.5f} · {b['n_sequences']} seq · {b['settings']['particles']} particles × "
              f"{b['settings']['sweeps']} sweeps · resets {d['reset_share']:.2%} · MH acceptance {d['mh_acceptance']:.2f} · "
              f"position-0 check: exact {c.get('exact', float('nan')):.5f} vs sampler {c.get('sampler_lo', float('nan')):.5f}"
              f"{'' if c.get('usable', False) else ' (exact check not usable at this ray count)'}")